In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("oddrationale/mnist-in-csv")

print("Path to dataset files:", path)

In [ ]:
## Step 2: Load the Dataset
import pandas as pd
import os

# Load test CSV file
file_path = os.path.join(path, 'mnist_test.csv')
df = pd.read_csv(file_path)

# Separate labels and features
y = df['label']
X = df.drop('label', axis=1)

df.head()


In [ ]:
## Step 3: Basic Statistical Analysis
# Mean, Median, Mode, Min, Max
mean = X.mean().mean()
median = X.median().median()
mode = X.mode().iloc[0].mode()
min_val = X.min().min()
max_val = X.max().max()

print(f"Mean: {mean}")
print(f"Median: {median}")
print(f"Mode: {mode}")
print(f"Min: {min_val}")
print(f"Max: {max_val}")


In [ ]:
## Step 4: Digit Distribution
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
sns.countplot(x=y)
plt.title("Digit Distribution")
plt.xlabel("Digit")
plt.ylabel("Frequency")
plt.grid(True)
plt.show()


In [ ]:
## Step 5: PCA Scatter Plot
from sklearn.decomposition import PCA

# Reduce dimensionality to 2D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# Plot
plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=y, palette="tab10", legend=False)
plt.title("PCA of MNIST Digits")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.grid(True)
plt.show()


In [ ]:
sampled_pixels = X.sample(n=500, random_state=42).values.flatten()
sns.histplot(sampled_pixels, bins=30, kde=True)


In [ ]:
## Step 6: Sample Digit Grids - Visualize 25 images per class
import numpy as np

def plot_digit_samples(X, y, digit, n_samples=25):
    """Plot a grid of sample images for a specific digit"""
    digit_data = X[y == digit]

    # Randomly sample n_samples images
    sample_indices = np.random.choice(len(digit_data), n_samples, replace=False)
    samples = digit_data.iloc[sample_indices]

    # Create subplot grid
    fig, axes = plt.subplots(5, 5, figsize=(8, 8))
    fig.suptitle(f'Sample Images of Digit {digit}', fontsize=16)

    for i, ax in enumerate(axes.flat):
        if i < len(samples):
            # Reshape 784 pixels back to 28x28 image
            image = samples.iloc[i].values.reshape(28, 28)
            ax.imshow(image, cmap='gray')
        ax.axis('off')

    plt.tight_layout()
    plt.show()

# Plot samples for each digit (0-9)
for digit in range(10):
    plot_digit_samples(X, y, digit)

## Step 7: Average Digit Image (Heatmap) - Mean pixel values per class
def plot_average_digits():
    """Create heatmaps showing average digit templates"""
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    fig.suptitle('Average Digit Templates (Heatmaps)', fontsize=16)

    for digit in range(10):
        row = digit // 5
        col = digit % 5

        # Get all images for this digit
        digit_data = X[y == digit]

        # Calculate mean image
        mean_image = digit_data.mean().values.reshape(28, 28)

        # Plot heatmap
        im = axes[row, col].imshow(mean_image, cmap='hot', interpolation='nearest')
        axes[row, col].set_title(f'Digit {digit}')
        axes[row, col].axis('off')

        # Add colorbar for reference
        plt.colorbar(im, ax=axes[row, col], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()

plot_average_digits()

## Step 8: Enhanced Pixel Intensity Analysis
def detailed_pixel_analysis():
    """More comprehensive pixel intensity analysis"""

    # Overall pixel distribution
    plt.figure(figsize=(15, 5))

    # Subplot 1: Overall histogram
    plt.subplot(1, 3, 1)
    sampled_pixels = X.sample(n=1000, random_state=42).values.flatten()
    plt.hist(sampled_pixels, bins=50, alpha=0.7, color='blue', edgecolor='black')
    plt.title('Overall Pixel Intensity Distribution')
    plt.xlabel('Pixel Value')
    plt.ylabel('Frequency')
    plt.grid(True, alpha=0.3)

    # Subplot 2: Box plot by digit
    plt.subplot(1, 3, 2)
    # Sample data for box plot (to avoid memory issues)
    sample_size = 100
    box_data = []
    labels = []
    for digit in range(10):
        digit_pixels = X[y == digit].sample(n=sample_size, random_state=42).values.flatten()
        box_data.append(digit_pixels)
        labels.append(str(digit))

    plt.boxplot(box_data, labels=labels)
    plt.title('Pixel Intensity by Digit')
    plt.xlabel('Digit')
    plt.ylabel('Pixel Value')
    plt.grid(True, alpha=0.3)

    # Subplot 3: Mean pixel intensity per digit
    plt.subplot(1, 3, 3)
    mean_intensities = [X[y == digit].values.mean() for digit in range(10)]
    plt.bar(range(10), mean_intensities, color='skyblue', edgecolor='black')
    plt.title('Average Pixel Intensity by Digit')
    plt.xlabel('Digit')
    plt.ylabel('Mean Pixel Value')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

detailed_pixel_analysis()

## Step 9: t-SNE Visualization (Advanced)
from sklearn.manifold import TSNE

def plot_tsne_visualization():
    """Create t-SNE visualization for better cluster separation"""

    # Sample data for t-SNE (it's computationally expensive)
    sample_size = 2000
    sample_indices = np.random.choice(len(X), sample_size, replace=False)
    X_sample = X.iloc[sample_indices]
    y_sample = y.iloc[sample_indices]

    print("Computing t-SNE... This may take a few minutes.")

    # Apply t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=30)
    X_tsne = tsne.fit_transform(X_sample)

    # Plot results
    plt.figure(figsize=(12, 5))

    # t-SNE plot
    plt.subplot(1, 2, 1)
    scatter = plt.scatter(X_tsne[:, 0], X_tsne[:, 1], c=y_sample, cmap='tab10', s=20, alpha=0.7)
    plt.colorbar(scatter)
    plt.title('t-SNE Visualization of MNIST Digits')
    plt.xlabel('t-SNE Component 1')
    plt.ylabel('t-SNE Component 2')
    plt.grid(True, alpha=0.3)

    # PCA vs t-SNE comparison (recompute PCA on same sample)
    pca_sample = PCA(n_components=2)
    X_pca_sample = pca_sample.fit_transform(X_sample)

    plt.subplot(1, 2, 2)
    scatter = plt.scatter(X_pca_sample[:, 0], X_pca_sample[:, 1], c=y_sample, cmap='tab10', s=20, alpha=0.7)
    plt.colorbar(scatter)
    plt.title('PCA Visualization (Same Sample)')
    plt.xlabel('Principal Component 1')
    plt.ylabel('Principal Component 2')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

plot_tsne_visualization()

## Step 10: Data Quality Assessment
def data_quality_analysis():
    """Analyze data quality and potential issues"""

    print("=== DATA QUALITY ANALYSIS ===\n")

    # Basic dataset info
    print(f"Dataset shape: {X.shape}")
    print(f"Number of features: {X.shape[1]} (28x28 pixels)")
    print(f"Number of samples: {X.shape[0]}")
    print(f"Data type: {X.dtypes[0]}")

    # Check for missing values
    missing_values = X.isnull().sum().sum()
    print(f"Missing values: {missing_values}")

    # Check value ranges
    print(f"Pixel value range: {X.min().min()} to {X.max().max()}")

    # Class distribution
    print("\nClass distribution:")
    class_counts = y.value_counts().sort_index()
    for digit, count in class_counts.items():
        percentage = (count / len(y)) * 100
        print(f"Digit {digit}: {count} samples ({percentage:.1f}%)")

    # Check for potential duplicates (computationally intensive, so sample)
    print(f"\nChecking for duplicates in sample of 1000 images...")
    sample_data = X.sample(n=1000, random_state=42)
    duplicates = sample_data.duplicated().sum()
    print(f"Duplicate images in sample: {duplicates}")

    # Analyze image properties
    print("\nImage property analysis:")

    # Images that are completely black
    black_images = (X.sum(axis=1) == 0).sum()
    print(f"Completely black images: {black_images}")

    # Images that are mostly white (very few non-zero pixels)
    sparse_threshold = 50  # Less than 50 non-zero pixels
    sparse_images = (X.astype(bool).sum(axis=1) < sparse_threshold).sum()
    print(f"Very sparse images (< {sparse_threshold} pixels): {sparse_images}")

    # Plot class balance
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.bar(class_counts.index, class_counts.values, color='lightblue', edgecolor='black')
    plt.title('Class Distribution')
    plt.xlabel('Digit')
    plt.ylabel('Count')
    plt.grid(True, alpha=0.3)

    plt.subplot(1, 2, 2)
    plt.pie(class_counts.values, labels=class_counts.index, autopct='%1.1f%%')
    plt.title('Class Distribution (Pie Chart)')

    plt.tight_layout()
    plt.show()

data_quality_analysis()

## Step 11: Feature Analysis - Most/Least Important Pixels
def pixel_importance_analysis():
    """Analyze which pixels are most informative"""

    # Calculate variance for each pixel across all images
    pixel_variances = X.var()

    # Reshape to 28x28 for visualization
    variance_map = pixel_variances.values.reshape(28, 28)

    plt.figure(figsize=(15, 5))

    # Variance heatmap
    plt.subplot(1, 3, 1)
    plt.imshow(variance_map, cmap='hot', interpolation='nearest')
    plt.title('Pixel Variance Across All Images')
    plt.colorbar()
    plt.axis('off')

    # Most variable pixels (most informative)
    plt.subplot(1, 3, 2)
    high_var_mask = variance_map > np.percentile(variance_map, 90)
    plt.imshow(high_var_mask, cmap='binary', interpolation='nearest')
    plt.title('Most Variable Pixels (Top 10%)')
    plt.axis('off')

    # Least variable pixels (least informative)
    plt.subplot(1, 3, 3)
    low_var_mask = variance_map < np.percentile(variance_map, 10)
    plt.imshow(low_var_mask, cmap='binary', interpolation='nearest')
    plt.title('Least Variable Pixels (Bottom 10%)')
    plt.axis('off')

    plt.tight_layout()
    plt.show()

    print(f"Most informative pixels are typically in the center region")
    print(f"Least informative pixels are typically around the edges")

pixel_importance_analysis()